# Burmese Agriculture CNER: Automated 5-Fold Cross-Validation Pipeline (With Auto-Resume)
This notebook is an enhanced, highly-professional version of the legacy Colab file. Instead of training on a single random split, it automates **5-Fold Cross-Validation** for all 4 standard CoNLL file formats (`bio_word`, `bio_syllable`, `bioes_word`, `bioes_syllable`), evaluates them using a standard entity-level metric, and reports the average Precision, Recall, and F1-Score along with their standard deviations.

### 💾 Colab Disconnect Protection & Auto-Resume
To protect against Google Colab timeouts, inactivity disconnects, or runtime restarts:
1. **Real-time Persistence**: The splits (`data/`), model checkpoints (`models/`), and prediction files (`output/`) are symlinked and written directly to your **Google Drive** in real-time.
2. **Auto-Resume**: If Colab disconnects, you can simply run all cells again. The training loop detects already completed folds and automatically skips them, parsing existing predictions to compute the final averages immediately. This saves days of training time!


### Step 1: Connect to Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


### Step 2: Configure Environment and Paths


In [ ]:
import os
import sys
import shutil

# Project directory in Google Drive
%env PJ_DIR=/content/drive/My Drive/MyAgriNER_copy
project_root = '/NCRFpp'

print(f'Project execution root is: {project_root}')


### Step 3: Clone NCRFpp Repository & Install Dependencies


In [ ]:
if not os.path.exists('/NCRFpp'):
    !git clone https://github.com/jiesutd/NCRFpp.git /NCRFpp
else:
    print('NCRFpp repository already exists.')

# Install the necessary libraries
with open('requirements.txt', 'w') as f:
    f.write('torch\nnumpy\n')
!pip install -r requirements.txt


### Step 4: Persistent Google Drive Symlinking & Dataset Extraction


In [ ]:
import os
import shutil

# 1. Define persistent directories in Google Drive
drive_root = '/content/drive/My Drive/MyAgriNER'
drive_data = os.path.join(drive_root, 'data')
drive_models = os.path.join(drive_root, 'models')
drive_output = os.path.join(drive_root, 'output')

# Create directories on Google Drive if they don't exist
os.makedirs(drive_data, exist_ok=True)
os.makedirs(drive_models, exist_ok=True)
os.makedirs(drive_output, exist_ok=True)

# 2. Extract data.zip directly to Google Drive (only if bio_word folder or emb files are missing to save time!)
already_extracted = os.path.exists(os.path.join(drive_data, 'bio_word')) and any(f.endswith('.emb') for f in os.listdir(drive_data)) if os.path.exists(drive_data) else False

if not already_extracted:
    zip_paths = [
        "/content/data.zip",
        "/content/drive/My Drive/MyAgriNER/data.zip",
        "/content/drive/My Drive/data.zip"
    ]
    found_zip = None
    for p in zip_paths:
        if os.path.exists(p):
            found_zip = p
            break
            
    if found_zip:
        print(f"Found data.zip at: {found_zip}")
        print("Extracting data.zip directly to Google Drive... This may take a minute.")
        !unzip -q -o "{found_zip}" -d "/content/drive/My Drive/MyAgriNER/"
        print("Extraction complete!")
    else:
        print("Warning: data.zip not found! If you have not uploaded pre-split folders, please upload data.zip to Google Drive (MyAgriNER/) or Colab (/content/).")
else:
    print("Data directory already exists and contains extracted files on Google Drive. Skipping extraction.")

# 3. Symlink /NCRFpp directories to Google Drive
for folder, drive_path in [('data', drive_data), ('models', drive_models), ('output', drive_output)]:
    local_path = f'/NCRFpp/{folder}'
    if os.path.exists(local_path):
        if os.path.islink(local_path):
            os.unlink(local_path)
        else:
            shutil.rmtree(local_path)
    os.symlink(drive_path, local_path)
    print(f'Symlinked local {local_path} -> persistent Google Drive: {drive_path}')


### Step 5: Robust K-Fold Dataset Splitter (Rotating Blocks)
Instead of random shuffling (which breaks cross-validation sequence standards), this cell splits each standard CoNLL file into 5 fold directories under `data/{setup_name}/fold_{fold_idx}/` containing standard `train.conll`, `dev.conll`, and `test.conll` files, respecting sentence boundaries.

**Smart Skipping**: It automatically checks your Google Drive, and if the split folders are already generated, it skips this process to save time.


In [ ]:
import os
from pathlib import Path

def load_conll_sentences(file_path):
    """Parses a CoNLL file and returns a list of raw sentence blocks."""
    sentences = []
    current_sentence = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            current_sentence.append(line)
            if not line.strip():
                if current_sentence:
                    sentences.append("".join(current_sentence))
                    current_sentence = []
        if current_sentence:
            content = "".join(current_sentence)
            if not content.endswith('\n'):
                content += '\n'
            if not content.endswith('\n\n'):
                content += '\n'
            sentences.append(content)
    return sentences

def split_sentences_by_ratio(sentences, ratio, folds):
    """Splits sentences into folds using rotating block selection."""
    B = sum(ratio)
    N = len(sentences)
    X, Y, Z = ratio
    
    folds_data = []
    for i in range(folds):
        S_i = int(i * B / folds)
        
        # Rotating block indices modulo B
        train_blocks = [(S_i + b) % B for b in range(X)]
        dev_blocks = [(S_i + X + b) % B for b in range(Y)]
        test_blocks = [(S_i + X + Y + b) % B for b in range(Z)]
        
        def get_sentences_for_blocks(blocks):
            selected = []
            for b in sorted(blocks):
                start_idx = int(b * N / B)
                end_idx = int((b + 1) * N / B)
                selected.extend(sentences[start_idx:end_idx])
            return selected
            
        train_sents = get_sentences_for_blocks(train_blocks)
        dev_sents = get_sentences_for_blocks(dev_blocks)
        test_sents = get_sentences_for_blocks(test_blocks)
        
        folds_data.append((train_sents, dev_sents, test_sents))
    return folds_data

def process_file(file_path, output_dir, ratio, folds):
    print(f"Processing CoNLL file: '{file_path}'")
    sentences = load_conll_sentences(file_path)
    print(f"  Loaded {len(sentences)} sentences.")
    
    folds_data = split_sentences_by_ratio(sentences, ratio, folds)
    base_name = Path(file_path).stem
    
    setup_dir = Path(output_dir) / base_name
    setup_dir.mkdir(parents=True, exist_ok=True)
    
    for i, (train, dev, test) in enumerate(folds_data):
        fold_dir = setup_dir / f"fold_{i}"
        fold_dir.mkdir(parents=True, exist_ok=True)
        
        with open(fold_dir / "train.conll", 'w', encoding='utf-8') as f:
            f.writelines(train)
        with open(fold_dir / "dev.conll", 'w', encoding='utf-8') as f:
            f.writelines(dev)
        with open(fold_dir / "test.conll", 'w', encoding='utf-8') as f:
            f.writelines(test)
            
        print(f"    Fold {i} -> Train: {len(train)} sents, Dev: {len(dev)} sents, Test: {len(test)} sents")

data_dir = '/NCRFpp/data'

# Check if splits are already present on Google Drive
has_splits = False
if os.path.exists(data_dir):
    subdirs = [os.path.join(data_dir, d) for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))]
    for subdir in subdirs:
        if 'fold_0' in os.listdir(subdir):
            has_splits = True
            break

if not has_splits:
    conll_files = [os.path.join(data_dir, f) for f in os.listdir(data_dir)
                   if f.endswith('.conll') and not f.startswith(('train.', 'dev.', 'test.'))]
    for file in sorted(conll_files):
        process_file(file, data_dir, [8, 1, 1], 5)
    print('\nAll datasets split into 5 folds successfully!')
else:
    print('Dataset splits already exist on Google Drive. Skipping splitting phase!')


### Step 6: Dynamic Training & Decoding Configuration Generator
This cell generates `.train.config` and `.decode.config` files for all configurations, including variations with and without fastText pre-trained embeddings.


In [ ]:
project_root = '/NCRFpp'

def get_emb_path(name):
    """Finds the most specific pretrained embedding file, with fallbacks."""
    candidates = [
        f"{project_root}/data/custom_burmese_agri_word.emb",
        f"{project_root}/data/burmese_agri_word.emb",
        f"{project_root}/data/custom_burmese_agri_syllable.emb",
        f"{project_root}/data/burmese_agri_syllable.emb",
    ]
    is_word = "word" in name.lower()
    is_syllable = "syllable" in name.lower()
    
    if is_word:
        for p in candidates:
            if "word" in p and os.path.exists(p):
                return p
        return f"{project_root}/data/burmese_agri_word.emb"
    elif is_syllable:
        for p in candidates:
            if "syllable" in p and os.path.exists(p):
                return p
        return f"{project_root}/data/burmese_agri_syllable.emb"
    return None

def create_fold_configs(setup_name, fold, with_emb=False):
    """Generates the .train.config configuration file for a given fold experiment."""
    config_prefix = "with_emb." if with_emb else ""
    config_name = f"{config_prefix}{setup_name}_fold_{fold}"
    
    word_emb_setting = ""
    if with_emb:
        emb_path = get_emb_path(setup_name)
        if emb_path:
            word_emb_setting = f"word_emb_dir={emb_path}"
            
    train_content = f"""
### use # to comment out the configure item

### I/O ###
train_dir={project_root}/data/{setup_name}/fold_{fold}/train.conll
dev_dir={project_root}/data/{setup_name}/fold_{fold}/dev.conll
test_dir={project_root}/data/{setup_name}/fold_{fold}/test.conll
model_dir={project_root}/models/{config_name}
{word_emb_setting}

norm_word_emb=False
norm_char_emb=False
number_normalized=True
seg=True
word_emb_dim=200
char_emb_dim=200

###NetworkConfiguration###
use_crf=True
use_char=True
word_seq_feature=LSTM
char_seq_feature=CNN

###TrainingSetting###
status=train
optimizer=ADAM
iteration=30
batch_size=10
ave_batch_loss=False

###Hyperparameters###
cnn_layer=4
char_hidden_dim=50
hidden_dim=200
dropout=0.3
lstm_layer=1
bilstm=True
learning_rate=0.001
lr_decay=0
momentum=0
l2=1e-8
gpu
clip=5.0
"""
    train_config_path = f"{project_root}/{config_name}.train.config"
    with open(train_config_path, 'w') as f:
        f.write(train_content.strip())

def create_fold_decode_config(config_name, setup_name, fold, model_path, dset_path):
    """Generates the .decode.config configuration file for evaluation."""
    decode_content = f"""
### I/O ###
status=decode
raw_dir={project_root}/data/{setup_name}/fold_{fold}/test.conll
decode_dir={project_root}/output/{config_name}.test.out
dset_dir={dset_path}
load_model_dir={model_path}

nbest=1
gpu
"""
    decode_config_path = f"{project_root}/{config_name}.decode.config"
    with open(decode_config_path, 'w') as f:
        f.write(decode_content.strip())

# Auto-generate configurations for all available setups
data_dir = '/NCRFpp/data'
setups = sorted([d for d in os.listdir(data_dir)
                 if os.path.isdir(os.path.join(data_dir, d)) and os.path.exists(os.path.join(data_dir, d, 'fold_0'))])

for setup in setups:
    for fold in range(5):
        create_fold_configs(setup, fold, with_emb=False)
        create_fold_configs(setup, fold, with_emb=True)

print(f"Successfully generated Training configurations for {len(setups)} setups across 5 folds!")


### Step 7: Standard CoNLL Entity-Level Performance Evaluator
We implement the official CoNLL entity-level metric (exact matches of type, start, and end token indices) in pure python. This avoids dependencies and ensures complete reliability of computed precision, recall, and F1 scores.


In [ ]:
def get_entities(tags):
    """Extracts standard entity triples (type, start_idx, end_idx) from tags list."""
    entities = []
    current_entity = None
    
    for i, tag in enumerate(tags):
        if tag == "O" or tag == "<pad>" or tag == "<unk>":
            if current_entity:
                entities.append(current_entity)
                current_entity = None
            continue
            
        if "-" in tag:
            boundary, entity_type = tag.split("-", 1)
        else:
            boundary = tag
            entity_type = "ENTITY"
            
        boundary = boundary.upper()
        
        if boundary == "B":
            if current_entity:
                entities.append(current_entity)
            current_entity = {"type": entity_type, "start": i, "end": i}
        elif boundary == "I":
            if current_entity and current_entity["type"] == entity_type:
                current_entity["end"] = i
            else:
                if current_entity:
                    entities.append(current_entity)
                current_entity = {"type": entity_type, "start": i, "end": i}
        elif boundary == "E":
            if current_entity and current_entity["type"] == entity_type:
                current_entity["end"] = i
                entities.append(current_entity)
                current_entity = None
            else:
                if current_entity:
                    entities.append(current_entity)
                entities.append({"type": entity_type, "start": i, "end": i})
                current_entity = None
        elif boundary == "S":
            if current_entity:
                entities.append(current_entity)
            entities.append({"type": entity_type, "start": i, "end": i})
            current_entity = None
            
    if current_entity:
        entities.append(current_entity)
        
    return set((ent["type"], ent["start"], ent["end"]) for ent in entities)

def evaluate_predictions(file_path):
    """Parses an NCRFpp predictions output file and computes entity metrics."""
    gold_tags = []
    pred_tags = []
    current_gold = []
    current_pred = []
    
    if not os.path.exists(file_path):
        return None
        
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                if current_gold:
                    gold_tags.append(current_gold)
                    pred_tags.append(current_pred)
                    current_gold = []
                    current_pred = []
                continue
            parts = line.split()
            if len(parts) >= 3:
                current_gold.append(parts[-2])
                current_pred.append(parts[-1])
            elif len(parts) == 2:
                current_gold.append(parts[0])
                current_pred.append(parts[1])
                
        if current_gold:
            gold_tags.append(current_gold)
            pred_tags.append(current_pred)
            
    total_gold_entities = 0
    total_pred_entities = 0
    correct_entities = 0
    
    for g_seq, p_seq in zip(gold_tags, pred_tags):
        g_ents = get_entities(g_seq)
        p_ents = get_entities(p_seq)
        
        total_gold_entities += len(g_ents)
        total_pred_entities += len(p_ents)
        correct_entities += len(g_ents & p_ents)
        
    precision = correct_entities / total_pred_entities if total_pred_entities > 0 else 0.0
    recall = correct_entities / total_gold_entities if total_gold_entities > 0 else 0.0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    
    return {
        "precision": precision * 100,
        "recall": recall * 100,
        "f1": f1 * 100,
        "gold_count": total_gold_entities,
        "pred_count": total_pred_entities,
        "correct_count": correct_entities
    }
            


### Step 8: Execution Master Loop - Train, Decode, Evaluate and Average Folds (With Checkpoint Resume)
This cell is the heart of the k-fold pipeline. It loops through your experiments, runs NCRFpp, evaluates the predictions using our custom evaluator, and compiles a comprehensive performance report.

**🔄 Automatic Resumption**:
- Before starting a fold's training, this script checks if the finished model (epoch 29) and the final predictions already exist on Google Drive.
- If they exist, it **instantly restores the predictions and evaluation metrics** and proceeds to the next fold.
- This protects you from Colab disconnected sessions: simply run this cell again, and it will pick up right where it left off, skipping already-completed experiments!


In [ ]:
import os
import subprocess
import glob
import numpy as np

# Auto-detect setups to run
setups = sorted([d for d in os.listdir('/NCRFpp/data')
                 if os.path.isdir(os.path.join('/NCRFpp/data', d)) and os.path.exists(os.path.join('/NCRFpp/data', d, 'fold_0'))])
variations = ["without_emb", "with_emb"]
num_folds = 5

# To test a single setup quickly, uncomment the line below:
# setups = ["first_sem_agri_bioes_word"]

results = {}
models_root = '/NCRFpp/models'
output_root = '/NCRFpp/output'

for setup in setups:
    results[setup] = {}
    for var in variations:
        results[setup][var] = []
        print("="*60)
        print(f"EXPERIMENT: {setup} ({var})")
        print("="*60)
        
        for fold in range(num_folds):
            config_prefix = "with_emb." if var == "with_emb" else ""
            config_name = f"{config_prefix}{setup}_fold_{fold}"
            
            # Check if this fold has already completed in a previous run
            final_epoch_model = os.path.join(models_root, f"{config_name}.29.model")
            pred_file = os.path.join(output_root, f"{config_name}.test.out")
            
            if os.path.exists(final_epoch_model) and os.path.exists(pred_file) and os.path.getsize(pred_file) > 0:
                print(f"\n[Fold {fold}] Already completed on Drive. Restoring predictions and skipping training.")
                metrics = evaluate_predictions(pred_file)
                if metrics:
                    print(f"  -> [Restored Results] Precision: {metrics['precision']:.2f}%, Recall: {metrics['recall']:.2f}%, F1: {metrics['f1']:.2f}%")
                    results[setup][var].append(metrics)
                continue
            
            # --- 1. Train ---
            train_config = f"/NCRFpp/{config_name}.train.config"
            print(f"\n[Fold {fold}] Training model...")
            train_cmd = ["python", "/NCRFpp/main.py", "--config", train_config]
            subprocess.run(train_cmd, check=True)
            
            # Find the saved models for this fold to select the best/latest epoch
            model_files = sorted(glob.glob(os.path.join(models_root, f"{config_name}.*.model")))
            dset_path = os.path.join(models_root, f"{config_name}.dset")
            
            if not model_files or not os.path.exists(dset_path):
                print(f"  Error: Training failed to produce models for {config_name}")
                continue
                
            best_model_path = model_files[-1]
            
            # --- 2. Generate Decode Config ---
            create_fold_decode_config(config_name, setup, fold, best_model_path, dset_path)
            
            # --- 3. Decode ---
            decode_config = f"/NCRFpp/{config_name}.decode.config"
            print(f"[Fold {fold}] Decoding predictions...")
            decode_cmd = ["python", "/NCRFpp/main.py", "--config", decode_config]
            subprocess.run(decode_cmd, check=True)
            
            # --- 4. Evaluate Predicted Output ---
            metrics = evaluate_predictions(pred_file)
            if metrics:
                print(f"  -> Fold {fold} Results: Precision: {metrics['precision']:.2f}%, Recall: {metrics['recall']:.2f}%, F1: {metrics['f1']:.2f}%")
                results[setup][var].append(metrics)
            else:
                print(f"  Error: Prediction file {pred_file} could not be analyzed.")

# --- 5. Generate and Display Cross-Validation Summary Report ---
print("\n" + "="*80)
print("             CROSS-VALIDATION PERFORMANCE REPORT (5 FOLDS)")
print("="*80)

for setup in setups:
    print(f"\nSetup: {setup}")
    print("-" * 50)
    for var in variations:
        fold_results = results[setup][var]
        if not fold_results:
            print(f"  * {var:11s} | No fold data available.")
            continue
            
        p_vals = [r["precision"] for r in fold_results]
        r_vals = [r["recall"] for r in fold_results]
        f_vals = [r["f1"] for r in fold_results]
        
        mean_p, std_p = np.mean(p_vals), np.std(p_vals)
        mean_r, std_r = np.mean(r_vals), np.std(r_vals)
        mean_f, std_f = np.mean(f_vals), np.std(f_vals)
        
        print(f"  * {var:11s} | Precision: {mean_p:5.2f}% ± {std_p:4.2f}% | Recall: {mean_r:5.2f}% ± {std_r:4.2f}% | F1: {mean_f:5.2f}% ± {std_f:4.2f}%")
print("="*80)


### Step 9: Backup All Experiments to Google Drive


In [ ]:
from datetime import datetime

# Define Google Drive export path
drive_export_path = '/content/drive/My Drive/MyAgriNER/'
ncrfpp_backup_path = os.path.join(drive_export_path, 'NCRFpp_backup')
os.makedirs(ncrfpp_backup_path, exist_ok=True)

src_ncrfpp_dir = '/NCRFpp'
dst_ncrfpp_dir = os.path.join(ncrfpp_backup_path, datetime.now().strftime('%Y%m%d_%H%M%S'))

print(f'Starting backup of {src_ncrfpp_dir} to {dst_ncrfpp_dir}...')

if os.path.exists(src_ncrfpp_dir):
    if os.path.exists(dst_ncrfpp_dir):
        shutil.rmtree(dst_ncrfpp_dir)
    shutil.copytree(src_ncrfpp_dir, dst_ncrfpp_dir)
    print(f'\nBackup complete! The entire directory is saved in {dst_ncrfpp_dir}.')
else:
    print(f'Error: Source directory {src_ncrfpp_dir} does not exist.')
